# Advanced Architectures and Applications in Agentic AI
## CA1 Assessment: Adaptive Customer Service Harness

Student name: Maximiliano Herrera </br>
Student ID: d00296830

## GLobal Flags

In [1]:
DELETE_IF_EXISTS_DATASET = True
PROTECT_EXPERIMENTS = False
TEST_TRAINING_CYCLE = False
TOP_N = 0
DATASET_NAME= 'CA1 - Customer Dataset'
DATASET_VERSION= 'v9'

## Dataset Preparation

### Load and Analysis

In [2]:
import pandas as pd

df = pd.read_csv('customer-dataset.csv')
df

,Unnamed: 0,flags,instruction,category,intent,response
0,0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...
...,...,...,...,...,...,...
26867,26867,BL,I am waiting for a rebate of {{Refund Amount}}...,REFUND,track_refund,Thank you for sharing your situation regarding...
26868,26868,BIL,how to see if there is anything wrong with my ...,REFUND,track_refund,Ensuring the accuracy of your restitution is o...
26869,26869,BLQZ,I'm waiting for a reimbjrsement of {{Currency ...,REFUND,track_refund,"Firstly, I genuinely understand the importance..."
26870,26870,BL,I don't know what to do to see my reimbursemen...,REFUND,track_refund,I've understood you're unsure about how to che...


In [3]:
categories = set(df['category'])
print(categories)
print(len(categories))

{'DELIVERY', 'SHIPPING', 'ACCOUNT', 'SUBSCRIPTION', 'FEEDBACK', 'PAYMENT', 'REFUND', 'CONTACT', 'CANCEL', 'INVOICE', 'ORDER'}
11


In [4]:
df.groupby(["category"]).size().reset_index(name="count")

,category,count
0,ACCOUNT,5986
1,CANCEL,950
2,CONTACT,1999
3,DELIVERY,1994
4,FEEDBACK,1997
5,INVOICE,1999
6,ORDER,3988
7,PAYMENT,1998
8,REFUND,2992
9,SHIPPING,1970


In [5]:
df.groupby(["category", "intent"]).size().reset_index(name="count")

,category,intent,count
0,ACCOUNT,create_account,997
1,ACCOUNT,delete_account,995
2,ACCOUNT,edit_account,1000
3,ACCOUNT,recover_password,995
4,ACCOUNT,registration_problems,999
5,ACCOUNT,switch_account,1000
6,CANCEL,check_cancellation_fee,950
7,CONTACT,contact_customer_service,1000
8,CONTACT,contact_human_agent,999
9,DELIVERY,delivery_options,995


### Filter dataset by categories and intents

In [6]:
# categories_selected = ['ACCOUNT', 'ORDER', 'REFUND']
# intents_selected = ['create_account', 'delete_account', 'edit_account',
#                     'cancel_order', 'change_order', 'place_order',
#                     'check_refund_policy', 'get_refund', 'track_refund'
#                    ]

categories_selected = ['ACCOUNT', 'ORDER', 'REFUND']
intents_selected = [
    'create_account','delete_account',  
    'cancel_order','place_order',
    'get_refund', 'track_refund'
]

df_filtered = df[
    df['category'].isin(categories_selected) &
    df['intent'].isin(intents_selected)
].rename(columns={"Unnamed: 0": "id"})

print(f"df_filtered: {len(df_filtered)}, {type(df_filtered)}")

df_filtered: 5983, <class 'pandas.DataFrame'>


In [7]:
import plotly.express as px

def plot_number_records_per_intent(dataframe):
    fig = px.bar(
        dataframe['intent'].value_counts().reset_index(),
        x='intent',
        y='count',
        title='Number of Records per Intent',
        text_auto=True
    )

    fig.show()

plot_number_records_per_intent(df_filtered)

In [8]:
from pandas import DataFrame

def get_split_dataset(categories_selected, intents_selected, max_total_rows=60, random_state=42):
    intent_count = len(intents_selected)
    # Largest even count per intent that stays within the cap, so train and test match.
    rows_per_intent = (max_total_rows // intent_count) // 2 * 2
    rows_per_split = rows_per_intent // 2
    if rows_per_split < 1:
        raise ValueError("max_total_rows is too small for one train and one test row per intent")

    print(f"intents: {intent_count}")
    print(f"rows_per_intent: {rows_per_intent}")
    print(f"rows_per_split: {rows_per_split}")

    ds_filtered = df[
        df["category"].isin(categories_selected) &
        df["intent"].isin(intents_selected)
    ].rename(columns={"Unnamed: 0": "id"})
    print(f"ds_filtered: {len(ds_filtered)}")

    sampled = ds_filtered.groupby("intent", group_keys=False).sample(
        n=rows_per_intent, random_state=random_state
    )
    train_df = sampled.groupby("intent", group_keys=False).head(rows_per_split)
    test_df = sampled.drop(train_df.index)

    print(f"sampled: {len(sampled)}")
    print(f"train_df: {len(train_df)}")
    print(f"test_df: {len(test_df)}")
    return train_df, test_df

def get_dataset_batches_unbalanced(dataframe: DataFrame, batch_size=30, random_state=42):
    shuffled_df = (
        dataframe
        .drop_duplicates(subset=["instruction"]) # remove duplicated instructions
        .sample(frac=1, random_state=random_state) # shuffle records and use random_state for reproducibility
        .reset_index(drop=True) # reset index and do not insert it as new column
    )

    for start in range(0, len(shuffled_df), batch_size): #get a sequence from start (0, inclusive) to stop (len(shuffled_df), exclusive) with a step (batch_size)
        batch = shuffled_df.iloc[start:start + batch_size] # select batch_size rows starting from start

        #if batch is incomplete then break the loop
        if len(batch) < batch_size: 
            break

        # return a copy of the batch so original data is unchanged
        yield batch.copy()


def get_dataset_batches_balanced(dataframe: DataFrame, batch_size=30, random_state=42):    
    remaining = dataframe.drop_duplicates(subset=["instruction"]).copy()    
    num_intents = remaining["intent"].nunique()  
    # Largest even count per intent that stays within the cap, so train and test match.
    rows_per_intent =  batch_size// num_intents

    if rows_per_intent == 0:
        raise ValueError("batch_size must be at least the number of intents")

    num_batches = remaining.groupby("intent").size().min() # rows per intent
    for _ in range(num_batches):
        batch = remaining.groupby("intent").sample(
            n=rows_per_intent,
            random_state=random_state
        )

        remaining= remaining.drop(batch.index)

        yield batch

def plot_and_preview_dataset(batch: DataFrame):
    plot_number_records_per_intent(batch)
    print(f"Num rows: {len(batch)}, num intents: {batch["intent"].nunique()}")
    print(len(batch))        
         

In [9]:
batches = get_dataset_batches_balanced(df_filtered,32)
train1_df =next(batches)
train2_df =next(batches)
test1_df =next(batches)
test2_df =next(batches)

if TOP_N:
    train1_df =train1_df.head(TOP_N)
    train2_df =train2_df.head(TOP_N)
    test1_df =test1_df.head(TOP_N)
    test2_df =test2_df.head(TOP_N)

# train_df, test_df = get_split_dataset(categories_selected, intents_selected, 50)

In [10]:
plot_and_preview_dataset(train1_df)
train1_df

Num rows: 30, num intents: 6
30


,id,flags,instruction,category,intent,response
860,860,BILZ,can you help me camceling the purchase I made?,ORDER,cancel_order,I've been informed that you need assistance wi...
79,79,BIQZ,whag do i need to do to cancel order {{Order N...,ORDER,cancel_order,I understand your need for assistance in cance...
479,479,BC,"I bought some product, I want to cancel order ...",ORDER,cancel_order,I've come to understand that you've made a pur...
209,209,BCELN,"I do not want this item, cancel order {{Order ...",ORDER,cancel_order,I've observed that you no longer want the item...
527,527,BCLN,I can no longer afford purchase {{Order Number...,ORDER,cancel_order,I completely understand the importance of canc...
10658,10658,BL,I need assistance to create a {{Account Catego...,ACCOUNT,create_account,Thank you for choosing to create a {{Account C...
10677,10677,BCLN,"I don't have a {{Account Category}} account, h...",ACCOUNT,create_account,Thank you so much for expressing your interest...
10581,10581,BCIL,"I ain't registered yet, how can I open a stand...",ACCOUNT,create_account,Thank you for expressing your interest in open...
9962,9962,BILQ,can ya help me to create a platinum account fo...,ACCOUNT,create_account,Indeed! I'm thrilled to assist you in creating...
10669,10669,BKLZ,opdn standard account,ACCOUNT,create_account,For sure! I'm here to guide you through the pr...


In [11]:
plot_and_preview_dataset(train2_df)
train2_df

Num rows: 30, num intents: 6
30


,id,flags,instruction,category,intent,response
78,78,BCL,I have got to cancel purchase {{Order Number}}...,ORDER,cancel_order,I've grasped that you need assistance with can...
321,321,BCILNPQ,i dont want purchase {{Order Number}} could i ...,ORDER,cancel_order,I grasp your desire to cancel your purchase wi...
801,801,BILZ,can you help me cancelling purchase {{Order N...,ORDER,cancel_order,I'll take care of it! I'm here to assist you w...
87,87,B,I need help cancelling order {{Order Number}},ORDER,cancel_order,I can relate to the understanding that you nee...
721,721,B,help canceling order {{Order Number}},ORDER,cancel_order,I catch on to the fact that you need assistanc...
10694,10694,BILQ,can i create a {{Account Category}} account fo...,ACCOUNT,create_account,I'm happy to help! It's wonderful to hear that...
10907,10907,BLZ,I have a problem with opening a standard account,ACCOUNT,create_account,I'm sorry to hear that you're experiencing dif...
10278,10278,BL,I need assistance opening a {{Account Category...,ACCOUNT,create_account,We appreciate your inquiry to us for assistanc...
10542,10542,BQ,i need help to create a platinum account,ACCOUNT,create_account,I'd be delighted to assist you in creating a {...
10343,10343,BLZ,I need assistance with opening a {{Account Typ...,ACCOUNT,create_account,"Wow, a {{Account Type}} account! That's an exc..."


In [12]:
plot_number_records_per_intent(test1_df)
test1_df

,id,flags,instruction,category,intent,response
758,758,BQZ,i want assisgance cancelling order {{Order Num...,ORDER,cancel_order,I've taken note that you're seeking assistance...
963,963,B,problem with canceling order {{Order Number}},ORDER,cancel_order,I realized that you're encountering difficulti...
222,222,BL,I want help cancelling purchase {{Order Number}},ORDER,cancel_order,"I'm on the same page, your desire for assistan..."
863,863,BLQ,need assistance to cancel purchase {{Order Num...,ORDER,cancel_order,I acknowledge that you're seeking assistance w...
459,459,BEPQZ,i do not know how i could casncel order {{Orde...,ORDER,cancel_order,I'm conscious of the reality that you're unsur...
10696,10696,BKLZ,creatingnew {{Account Category}} account,ACCOUNT,create_account,That's great news! I'm thrilled to assist you ...
10577,10577,BILQ,how to open a {{Account Type}} account for my ...,ACCOUNT,create_account,That's fantastic! I'm thrilled to assist you i...
10279,10279,BKL,new {{Account Type}} account for wife,ACCOUNT,create_account,How exciting to hear that you want to open a n...
10543,10543,BIL,how can I get information about opening a stan...,ACCOUNT,create_account,Thank you for expressing interest in opening a...
10345,10345,BCELNQZ,i do nothave a {{Account Type}} account help s...,ACCOUNT,create_account,Appreciate you reaching out to us! I'm excited...


In [13]:
plot_number_records_per_intent(test2_df)
test2_df

,id,flags,instruction,category,intent,response
964,964,BQ,need help cancelling order {{Order Number}},ORDER,cancel_order,I've become conscious that you're seeking assi...
34,34,BQ,need assistance to cancel order {{Order Number}},ORDER,cancel_order,I've got that you're seeking assistance in can...
158,158,BILQ,where do i cancel purchase {{Order Number}},ORDER,cancel_order,I'm cognizant of the fact that you're looking ...
80,80,B,I want assistance canceling order {{Order Numb...,ORDER,cancel_order,I've realized that you're seeking assistance w...
89,89,BLQ,i want help to cancel purchase {{Order Number}},ORDER,cancel_order,I completely understand that you need assistan...
10430,10430,BLM,find information about opening {{Account Categ...,ACCOUNT,create_account,I appreciate your interest in gathering inform...
10552,10552,BKLZ,new gold acount for wife,ACCOUNT,create_account,Thank you for considering opening a new {{Acco...
10505,10505,BLM,I have problems with opening my {{Account Type...,ACCOUNT,create_account,I apologize for the inconvenience you have fac...
10726,10726,BILPQZ,would it be possible to oen a gold account,ACCOUNT,create_account,I'm delighted that you're considering opening ...
10531,10531,BKL,create new pro account for daughter,ACCOUNT,create_account,That's wonderful! I'm thrilled to assist you i...


## Test Harnes Agent

In [14]:
from general import clean_agentic_memory

clean_agentic_memory()

### Test inference mode, wihtout agentic memory

In [15]:
import logging
from general import initialize_logging
from harness_agent.harness import AgentHarness
from llm import get_llm

initialize_logging(level=logging.INFO)

# model = get_llm("qwen/qwen3.8-27b")
# model = get_llm("openai/gpt-oss-120b")
# model = get_llm("gpt-6-astra")
# model = get_llm("gpt-6-luna-low")
model = get_llm("deepseek-flash-low")
 
harness = AgentHarness(model = model, training_dataset=train1_df, training_mode=False, agentic_memory_disabled=True)
# # response = harness.invoke("see the available payment options")
# # response = harness.invoke("were can i cancel purchase {{Order Number}}")
response = harness.invoke("can you help me camceling the purchase I made?")
response

2026-11-04 03:05:31 - harness.py - __post_init__ - INFO - Initializing AgentHarness - training_mode: False, agentic_memory_disabled: True
2026-11-04 03:05:32 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:34 - harness.py - invoke - INFO - Harness Response: I can help you with cancelling your purchase. To get started, please provide:

- The order number (or confirmation email reference)
- The item/service purchased
- The purchase date
- The platform/website where you bought it
- Whether it has already shipped or been delivered

Please don’t share full payment card numbers or sensitive passwords. If you tell me these details, I’ll guide you through the cancellation steps or help draft a cancellation request.


{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='1b6f668b-453b-4fcb-894f-2ac868fef897'),
  AIMessage(content='I can help you with cancelling your purchase. To get started, please provide:\n\n- The order number (or confirmation email reference)\n- The item/service purchased\n- The purchase date\n- The platform/website where you bought it\n- Whether it has already shipped or been delivered\n\nPlease don’t share full payment card numbers or sensitive passwords. If you tell me these details, I’ll guide you through the cancellation steps or help draft a cancellation request.', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer. Need help cancel purchase. Need ask details? We are customer service assistant but no context. Need clarify platform/order info. Need not ask sensitive? We can guide. Need say can help. Need know order number, item, date, reason, whether shipped, platform. If pur

### Test multi-turn

In [16]:
response = harness.invoke("Unfortunatelly I cannot give you none of those details...")
response

2026-11-04 03:05:34 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:39 - harness.py - invoke - INFO - Harness Response: No problem — you don’t need to share order details with me. I can’t directly access or cancel a purchase for you, but I can help you do it safely.

Here’s the fastest route without those details:

1. **Search your email** for “order confirmation,” “receipt,” or the merchant name. The cancellation link/order number is usually there.
2. **Check the merchant’s website** → Sign in → Order history / Purchases → Cancel.
3. **Use the merchant’s live chat/contact form** and say:  
   > “I’d like to cancel my recent purchase. I don’t have the order number available. Please help identify the order using my email/account and confirm cancellation and refund.”
4. **If it’s a subscription**, go to Account → Billing/Subscriptions → Cancel.
5. **If it already shipped**, request a return/re

{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='1b6f668b-453b-4fcb-894f-2ac868fef897'),
  AIMessage(content='I can help you with cancelling your purchase. To get started, please provide:\n\n- The order number (or confirmation email reference)\n- The item/service purchased\n- The purchase date\n- The platform/website where you bought it\n- Whether it has already shipped or been delivered\n\nPlease don’t share full payment card numbers or sensitive passwords. If you tell me these details, I’ll guide you through the cancellation steps or help draft a cancellation request.', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer. Need help cancel purchase. Need ask details? We are customer service assistant but no context. Need clarify platform/order info. Need not ask sensitive? We can guide. Need say can help. Need know order number, item, date, reason, whether shipped, platform. If pur

In [17]:
response = harness.invoke("Can you remind me my first question please?")
response

2026-11-04 03:05:39 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:40 - harness.py - invoke - INFO - Harness Response: Your first question was:

**“can you help me camceling the purchase I made?”**

(You meant “canceling.”)


{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='1b6f668b-453b-4fcb-894f-2ac868fef897'),
  AIMessage(content='I can help you with cancelling your purchase. To get started, please provide:\n\n- The order number (or confirmation email reference)\n- The item/service purchased\n- The purchase date\n- The platform/website where you bought it\n- Whether it has already shipped or been delivered\n\nPlease don’t share full payment card numbers or sensitive passwords. If you tell me these details, I’ll guide you through the cancellation steps or help draft a cancellation request.', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer. Need help cancel purchase. Need ask details? We are customer service assistant but no context. Need clarify platform/order info. Need not ask sensitive? We can guide. Need say can help. Need know order number, item, date, reason, whether shipped, platform. If pur

### Test training mode and enable agentic memory

In [18]:
from harness_agent.harness import AgentHarness
import logging
from llm import get_llm

initialize_logging(level=logging.INFO)

# model = get_llm("qwen/qwen3.8-27b")
model = get_llm("openai/gpt-oss-120b")
# model = get_llm("gpt-6-luna")
harness = AgentHarness(model = model, training_dataset=train1_df, training_mode=True, agentic_memory_disabled=False)
response = harness.invoke("can you help me camceling the purchase I made?")
response

2026-11-04 03:05:40 - harness.py - __post_init__ - INFO - Initializing AgentHarness - training_mode: True, agentic_memory_disabled: False
2026-11-04 03:05:40 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:40 - supervisor.py - request_supervisor - INFO - Supervisor called
2026-11-04 03:05:41 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:42 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:42 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:42 - skills_creator_agent.py - invoke - INFO - SkillsCreatorAgent executed successfully, number of tool calls: 1
2026-11-04 03:05:43 - _client.py - _send_single_

{'messages': [HumanMessage(content='can you help me camceling the purchase I made?', additional_kwargs={}, response_metadata={}, id='bb6d2073-3c2b-4377-a364-322719b908e4'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants help canceling a purchase. No skills are listed. According to instructions, if no skill matches, call request_supervisor before answering. So we need to call request_supervisor with user_query.', 'tool_calls': [{'id': 'fc_0389cce6-1c83-40c5-bc05-f25d8149c156', 'function': {'arguments': '{"user_query":"can you help me camceling the purchase I made?"}', 'name': 'request_supervisor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 1021, 'total_tokens': 1103, 'completion_time': 0.170709911, 'completion_tokens_details': {'reasoning_tokens': 43}, 'prompt_time': 0.000308123, 'prompt_tokens_details': {'cached_tokens': 509}, 'queue_time': 0.017509883, 'total_time': 0.171018034}, 'model_name': '

### Test that multiturn should be disable on training mode

In [19]:
response = harness.invoke("What was my first question?")
response

2026-11-04 03:05:45 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:45 - supervisor.py - request_supervisor - INFO - Supervisor called
2026-11-04 03:05:45 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:46 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:46 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:46 - skills_creator_agent.py - invoke - INFO - SkillsCreatorAgent executed successfully, number of tool calls: 1
2026-11-04 03:05:48 - _client.py - _send_single_request - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-11-04 03:05:49 - _client.py - _

{'messages': [HumanMessage(content='What was my first question?', additional_kwargs={}, response_metadata={}, id='8c1c9e44-31cb-4194-84ec-135c940ab903'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What was my first question?" This is a request for conversation history. The assistant does not have that info; we could ask for clarification. There\'s no skill matching. According to guidelines, if no skill matches, we should request supervisor before answering. So we need to call request_supervisor with the user query.', 'tool_calls': [{'id': 'fc_4717d8b2-cc92-4e46-bd6c-b9b9436e7a10', 'function': {'arguments': '{"user_query":"What was my first question?"}', 'name': 'request_supervisor'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 101, 'prompt_tokens': 717, 'total_tokens': 818, 'completion_time': 0.221414833, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.030151985, 'prompt_tokens_details': {'

In [20]:
clean_agentic_memory()

2026-11-04 03:05:49 - general.py - clean_agentic_memory - INFO - Cleaning agentic memory...
2026-11-04 03:05:49 - general.py - clean_agentic_memory - INFO -  * rmtree Skills...
2026-11-04 03:05:49 - general.py - clean_agentic_memory - INFO -  * .skills folder created...
2026-11-04 03:05:49 - general.py - clean_agentic_memory - INFO -  * Deleted AGENT.md...
2026-11-04 03:05:49 - general.py - clean_agentic_memory - INFO -  * ./AGENT.md file created...


## Testing and Evaluation

### Create Dataset and examples

In [21]:

from langsmith import Client
from langsmith.schemas import Dataset
from langsmith.utils import LangSmithConflictError

client = Client()

def create_dataset(name, description, delete_if_exists= False):
    try:        
        if client.has_dataset(dataset_name=name):
            if delete_if_exists:
                print(f"Dataset '{name}' already exists. Deleting existing dataset...")
                client.delete_dataset(dataset_name=name)
            else:
                print(f"Dataset '{name}' already exists. Skipping and retrieving from Langsmith...")
                dataset = client.read_dataset(dataset_name=name)
                return dataset

        dataset = client.create_dataset(dataset_name = name, description = description)
        print(f"Dataset '{name}' created successfully.")
        
        return dataset       
    except LangSmithConflictError as e:
        print(f"Error on creating Dataset '{name}': {str(e)}")  

dataset_name = f'{DATASET_NAME} - {DATASET_VERSION}'

dataset = create_dataset(
    name = dataset_name,
    description='Customer questions and responses',
    delete_if_exists= DELETE_IF_EXISTS_DATASET
)

Dataset 'CA1 - Customer Dataset - v9' already exists. Deleting existing dataset...
Dataset 'CA1 - Customer Dataset - v9' created successfully.


In [22]:
from pandas import DataFrame

def create_examples(dataset: Dataset, df: DataFrame, split_name: str = 'train'):
    if PROTECT_EXPERIMENTS:
        print("PROTECT_EXPERIMENTS enabled, skipping...")
        return
    
    print(f"Creating {len(df)} examples for split: {split_name}")

    example_ids = []
    for _,row in df.iterrows():
        example = client.create_example(
            dataset_id=dataset.id,            
            inputs= {
                "id": row["id"],
                "category": row["category"],
                "intent": row["intent"],
                "instruction": row["instruction"],
            },
            outputs= {
                "response": row["response"]                         
            },
            metadata={
                "category": row["category"],
                "intent": row["intent"],
            }
        )

        example_ids.append(example.id)

    print(f"Set split name: {split_name}")
    client.update_examples(example_ids=example_ids,splits=[split_name] * len(example_ids))

In [23]:
SPLIT_TRAIN_1 = 'train_1'
SPLIT_TRAIN_2 = 'train_2'
SPLIT_TEST_1 = 'test_1'
SPLIT_TEST_2 = 'test_2'
create_examples(dataset, train1_df, SPLIT_TRAIN_1)
create_examples(dataset, train2_df, SPLIT_TRAIN_2)
create_examples(dataset, test1_df, SPLIT_TEST_1)
create_examples(dataset, test2_df, SPLIT_TEST_2)

Creating 30 examples for split: train_1
Set split name: train_1
Creating 30 examples for split: train_2
Set split name: train_2
Creating 30 examples for split: test_1
Set split name: test_1
Creating 30 examples for split: test_2
Set split name: test_2


In [24]:
import pandas as pd

def get_examples_by_split(dataset_name, split_name):    
    examples = list(
        client.list_examples(
            dataset_name=dataset_name,
            splits=[split_name]
        )
    )

    df_examples = pd.DataFrame([
        {
            **example.inputs, #unpack operator
            **example.outputs,
            "langsmith_id": str(example.id),
            "splits": example.metadata.get("dataset_split")
                    if example.metadata else None
        }
        for example in examples
    ])

    return {
        'name' : split_name,
        'examples' : examples,
        'df': df_examples
    }

split_train_1 = get_examples_by_split(dataset_name,SPLIT_TRAIN_1)
split_train_2 = get_examples_by_split(dataset_name,SPLIT_TRAIN_2)
split_test_1 = get_examples_by_split(dataset_name,SPLIT_TEST_1)
split_test_2 = get_examples_by_split(dataset_name,SPLIT_TEST_2)

### Define LLM-as-Judge: evaluator_agent_response_quality

In [25]:
from harness_agent.harness import AgentHarness
from pydantic import BaseModel, Field
from llm import get_llm
from langchain_core.messages import HumanMessage, SystemMessage

class ResponseEvaluatorSchema(BaseModel):
    """Class to define the schema for quality evaluation."""
    score: int = Field(description="An integer score from 1 to 10 indicating the quality of the generated answer.")
    comment: str = Field(description="A brief explanation of the score")

def evaluator_agent_response_quality(inputs: dict, outputs: dict, reference_outputs: dict, model_name:str) -> dict:
    """Evaluate the quality of the agent's generated response using LLM with a structured output."""  
    query = inputs["instruction"]
    agent_response = outputs['agent_response']
    reference_response = reference_outputs['response']

    system_eval_prompt = "You score customer service answers."
    eval_prompt = f"""Score the agent answer from 1 to 10 against the reference.

1 means it does not match.
5 means some of it matches, but important parts are missing.
10 means it is the same kind of answer and the important information is there.

Do not give a high score just because it is polite.
If the reference has steps or placeholders, the answer should have them too.
If the answer makes up a procedure the reference does not have, score it lower.
Write a short comment on what matched and what was missing.

User query:
{query}

Reference answer:
{reference_response}

Agent response:
{agent_response}"""

    model = get_llm(model_name)
    result = model.with_structured_output(ResponseEvaluatorSchema).invoke([SystemMessage(system_eval_prompt), HumanMessage(eval_prompt)])
    
    evaluator_result = {
        "key": "response_quality",
        "score": result.score, 
        "comment": result.comment
        }
    logging.debug(f"evaluator_agent_response_quality: {evaluator_result}")

    return evaluator_result

### Evaluation Pipeline

In [26]:
def run_agent(query:str, model_name: str, training_dataset, training_mode: bool, agentic_memory_disabled : bool):      
    model = get_llm(model_name)
    harness = AgentHarness(model = model, training_dataset=training_dataset, training_mode=training_mode, agentic_memory_disabled=agentic_memory_disabled)
    # response = harness.invoke("see the available payment options")
    response = harness.invoke(query)
    agent_response = response['text']
    
    logging.debug("---------------------run_agent------------------------")
    logging.debug(f"response: {agent_response}")
    
    return agent_response

def target_function(inputs: str, model_name: str, training_dataset, training_mode: bool, agentic_memory_disabled : bool) -> dict:
    agent_response = run_agent(inputs["instruction"], model_name, training_dataset, training_mode, agentic_memory_disabled)
    logging.debug("---------------------target_function------------------------")
    logging.debug(f"agent_response: {agent_response}")
    
    return {"agent_response": agent_response}

def evaluate_agent(
    agent_model_name: str,
    judge_model_name: str,
    training: bool,
    disable_agentic_memory: bool,
    split: dict,
    stage: str,    
    run_evaluators: bool = True,
    ):
    
    #avoid answers leakage during test
    dataframe = split['df'] if training else None
    examples = split['examples']
    training_mode = "training" if training else "testing"
    agentic_memory_mode  = "memory_disabled" if disable_agentic_memory else "memory_on"
    agent_model_name_sanitized = agent_model_name.replace("/", "-")
    
    logging.debug(f"Evaluate Agent --> mode: {training_mode}, agentic_memory_mode: {agentic_memory_mode}")
    
    evaluators = []    
    if run_evaluators: 
         evaluators=[lambda inputs, outputs, reference_outputs : evaluator_agent_response_quality( inputs, outputs, reference_outputs, judge_model_name )]
         
    results =  client.evaluate(
        lambda inputs: target_function(inputs, agent_model_name, dataframe, training, disable_agentic_memory),
        data= examples,
        evaluators= evaluators,
        experiment_prefix=f"{stage}-{agent_model_name_sanitized}",       
         metadata={
            "stage": stage,           
            "model_name": agent_model_name,
            "judge_model": judge_model_name,
            "training_mode": training_mode,
            "agentic_memory": agentic_memory_mode,
            "split": split["name"],
            "n_examples": len(examples)
        },
        #Training must run sequentially because there are read/write operations into files
        max_concurrency= 1 if training else 5 ,
        #Ensure that evaluate blocks until evaluation completes
        blocking=True, 
    )
    
    return results    

In [27]:
from general import backup_agentic_memory, clean_agentic_memory
    
def evaluation_pipeline(agent_models, judge_model, run_evaluators_on_training = False):
    results_by_model = {}
    
    for model in agent_models:
        logging.info(f"Executing evaluation pipeline for model: {model}...")   
        clean_agentic_memory()   
        
        model_results = {}
        #Baseline: Evaluate agent on inference mode with agentic memory disabled
        model_results["baseline"] = evaluate_agent(model, judge_model, training=False, disable_agentic_memory=True, split=split_test_1, stage="baseline")
        
        #Training 1: Evaluate agent on training mode with agentic memory enabled        
        model_results["train_1"] = evaluate_agent(model, judge_model, training=True, disable_agentic_memory=False, split=split_train_1, stage="train_1", run_evaluators=run_evaluators_on_training)
        
        #Test 1 (after training 1): Evaluate agent on inference mode with agentic memory enabled using split_test_1 
        model_results["post_train_1"] = evaluate_agent(model, judge_model, training=False, disable_agentic_memory=False, split=split_test_1, stage="post_train_1")

        #Training 2: Evaluate agent on training mode with agentic memory enabled        
        model_results["train_2"] = evaluate_agent(model, judge_model, training=True, disable_agentic_memory=False, split=split_train_2, stage="train_2", run_evaluators=run_evaluators_on_training)
        
        #Test 2 (after training 2): Evaluate agent on inference mode with agentic memory enabled using split_test_1 
        model_results["post_train_2"] = evaluate_agent(model, judge_model, training=False, disable_agentic_memory=False, split=split_test_1, stage="post_train_2")
        
        #Test 3 (after training 2): Evaluate agent on inference mode with agentic memory enabled using split_test_2
        model_results["final_test"] = evaluate_agent(model, judge_model, training=False, disable_agentic_memory=False, split=split_test_2, stage="final_test")
        
        results_by_model[model]=model_results
        
        backup_agentic_memory(DATASET_VERSION, model)
        
        
    return results_by_model

### Test a training cycle offline (No LangSmith evaluations)

In [28]:
if TEST_TRAINING_CYCLE:
    # model_name="openai/gpt-oss-120b"
    # model_name="deepseek-flash-low"
    model_name="gpt-6-luna-low"
    # model_name="gpt-6-luna-medium"
        
    def train_agent_offline(model_name, split):
        dataframe:DataFrame =split['df']
        
        for row in dataframe.itertuples():
            print(row)
            agent_response = run_agent(query= row.instruction, model_name = model_name, training_dataset= dataframe, training_mode =True, agentic_memory_disabled=False)
        
    clean_agentic_memory()
    initialize_logging(level=logging.ERROR)

    train_agent_offline(model_name,split_train_1)    
else: 
    print("TEST_TRAINING_CYCLE disabled, skipping...")

TEST_TRAINING_CYCLE disabled, skipping...


In [29]:
if PROTECT_EXPERIMENTS:
    raise SystemError("PROTECT_EXPERIMENTS enabled, break...")

In [30]:
initialize_logging(logging.WARNING)

AGENT_MODELS = [
    "openai/gpt-oss-120b",
    "deepseek-flash-low",
    "gpt-6-luna-low",
    "gpt-6-luna-medium",
]

JUDGE_MODEL = "gpt-5-nano" # "deepseek-v4-pro" failing on parsing structured output
JUDGE_MODEL = "gpt-5.5" # better results
JUDGE_MODEL = "gpt-6-luna-low" # more critic
# JUDGE_MODEL = "gpt-6-luna-medium"  # more critic but slower
results = evaluation_pipeline(agent_models=AGENT_MODELS,judge_model= JUDGE_MODEL,run_evaluators_on_training= True)

View the evaluation results for experiment: 'baseline-openai-gpt-oss-120b-dce42601' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=1bdef2e6-60b1-4325-a3c1-ab2ecd41f342


View the evaluation results for experiment: 'train_1-openai-gpt-oss-120b-575d6c31' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=561e6b3e-d99d-44a9-ab97-76d5ff4c9724


View the evaluation results for experiment: 'post_train_1-openai-gpt-oss-120b-aa183604' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=4e1b3fdc-5956-4d78-964b-1d3521f794a7


View the evaluation results for experiment: 'train_2-openai-gpt-oss-120b-179ba21f' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be1

2026-11-04 03:11:29 - general.py - backup_agentic_memory - WARNING - Copying agentic memory into: backup\v9\openai-gpt-oss-120b


View the evaluation results for experiment: 'baseline-deepseek-flash-low-cbe640c5' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=2e16f663-c24d-4ae1-bdf6-1d9cc58f5b76


View the evaluation results for experiment: 'train_1-deepseek-flash-low-873e2ca0' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=c6629fe6-a3d9-4d9e-bfb3-d9ef4484561b


View the evaluation results for experiment: 'post_train_1-deepseek-flash-low-13aa3f37' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=ac782f6c-9f2e-44fd-9c7c-92e9492824f8


View the evaluation results for experiment: 'train_2-deepseek-flash-low-b97000b6' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb

2026-11-04 03:24:37 - general.py - backup_agentic_memory - WARNING - Copying agentic memory into: backup\v9\deepseek-flash-low


View the evaluation results for experiment: 'baseline-gpt-6-luna-low-0e67e365' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=276fb173-da05-4f42-bec7-85ed33989ec6


View the evaluation results for experiment: 'train_1-gpt-6-luna-low-82b74ef5' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=7787e93d-a035-44de-9505-d65fd01ded9d


View the evaluation results for experiment: 'post_train_1-gpt-6-luna-low-4da1155c' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=362e2493-e637-4130-acad-1224fb96093b


View the evaluation results for experiment: 'train_2-gpt-6-luna-low-a881a01a' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?se

2026-11-04 09:32:30 - general.py - backup_agentic_memory - WARNING - Copying agentic memory into: backup\v9\gpt-6-luna-low


View the evaluation results for experiment: 'baseline-gpt-6-luna-medium-49b707f4' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=f8b1b892-33cd-4da8-8520-8b9574a656f3


View the evaluation results for experiment: 'train_1-gpt-6-luna-medium-2a35e87e' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=dd672f02-47ea-4758-b29a-9245611c75f2


View the evaluation results for experiment: 'post_train_1-gpt-6-luna-medium-2fe1ec85' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab0/compare?selectedSessions=e57f853b-5742-4027-9c15-5f20d498c098


View the evaluation results for experiment: 'train_2-gpt-6-luna-medium-38175b1f' at:
https://eu.smith.langchain.com/o/0825fd4b-637a-48b4-9b2f-3c941d315004/datasets/9af3b5a2-c2d5-4df1-bd3b-be18fbb8bab

2026-11-04 09:47:22 - general.py - backup_agentic_memory - WARNING - Copying agentic memory into: backup\v9\gpt-6-luna-medium


In [31]:
results

{'openai/gpt-oss-120b': {'baseline': <ExperimentResults baseline-openai-gpt-oss-120b-dce42601>,
  'train_1': <ExperimentResults train_1-openai-gpt-oss-120b-575d6c31>,
  'post_train_1': <ExperimentResults post_train_1-openai-gpt-oss-120b-aa183604>,
  'train_2': <ExperimentResults train_2-openai-gpt-oss-120b-179ba21f>,
  'post_train_2': <ExperimentResults post_train_2-openai-gpt-oss-120b-df7644d0>,
  'final_test': <ExperimentResults final_test-openai-gpt-oss-120b-18ce6453>},
 'deepseek-flash-low': {'baseline': <ExperimentResults baseline-deepseek-flash-low-cbe640c5>,
  'train_1': <ExperimentResults train_1-deepseek-flash-low-873e2ca0>,
  'post_train_1': <ExperimentResults post_train_1-deepseek-flash-low-13aa3f37>,
  'train_2': <ExperimentResults train_2-deepseek-flash-low-b97000b6>,
  'post_train_2': <ExperimentResults post_train_2-deepseek-flash-low-8319f82b>,
  'final_test': <ExperimentResults final_test-deepseek-flash-low-458a6bfe>},
 'gpt-6-luna-low': {'baseline': <ExperimentResults 

In [ ]:
df_results = pd.DataFrame(results)
df_results

,openai/gpt-oss-120b,deepseek-flash-low,gpt-6-luna-low,gpt-6-luna-medium
baseline,({'run': id=UUID('01a1a485-582e-74f1-b74b-71e1...,({'run': id=UUID('01a1a48a-4c61-7d33-8f96-bf3a...,({'run': id=UUID('01a1a496-5452-7f01-ac2d-4f28...,({'run': id=UUID('01a1a5e7-226c-7812-af1a-bed3...
train_1,({'run': id=UUID('01a1a485-9119-7761-951f-76e2...,({'run': id=UUID('01a1a48a-a50c-79e0-a476-7ca3...,({'run': id=UUID('01a1a496-88b0-7792-8549-c315...,({'run': id=UUID('01a1a5e7-6022-7280-91d6-5cd9...
post_train_1,({'run': id=UUID('01a1a487-b8da-7172-9faa-035c...,({'run': id=UUID('01a1a48f-ceaa-7161-bfb7-6122...,({'run': id=UUID('01a1a5e1-00b3-72f1-b07c-9922...,({'run': id=UUID('01a1a5ed-49cb-7eb1-9a94-f3d5...
train_2,({'run': id=UUID('01a1a487-eb90-7250-b83a-5806...,({'run': id=UUID('01a1a490-2fed-7ce3-9243-0388...,({'run': id=UUID('01a1a5e1-59af-7c63-8921-32fe...,({'run': id=UUID('01a1a5ed-af15-7660-b8b0-7a75...
post_train_2,({'run': id=UUID('01a1a489-d7ff-7080-9251-66f0...,({'run': id=UUID('01a1a495-a3e5-7373-a6b2-6dbd...,({'run': id=UUID('01a1a5e6-645e-74e3-963e-e41e...,({'run': id=UUID('01a1a5f3-eed3-7b42-b4a1-4bba...
final_test,({'run': id=UUID('01a1a48a-1713-7883-8404-af19...,({'run': id=UUID('01a1a495-fbea-7c92-9975-d244...,({'run': id=UUID('01a1a5e6-c0ad-76a1-ab02-296d...,({'run': id=UUID('01a1a5f4-53e6-78a2-a91a-c27b...


: 